# Trabajo Práctico Final: Visualización de Datos
## Visualización de Datos utilizando la World Bank Data360

**Curso:** Visualización de la Información  
**Tema:** Análisis de Inversión y Resultados Educativos mediante World Bank Data360 API

---

### Objetivos del Proyecto
Explorar la API de datos del Banco Mundial para identificar bases de datos e indicadores disponibles.

* **Exploración de Bases de Datos:** Identificación de las bases de datos existentes en la plataforma.
* **Identificación de Indicadores:** Relevamiento de indicadores disponibles para el análisis.
* **Estructura de Datos:** Análisis de la organización de la información (países, series temporales, temáticas, etc.).

---

### Integrantes
* **Schelp**, Martin
* **Cozzi**, Alejandro Luis
* **Blanco Villar**, Maximiliano Jesús

In [1]:
%pip install -q "vegafusion[embed]>=1.5.0" "vegafusion>=1.5.0" "vl-convert-python>=1.6.0"
%pip install pandas
%pip install altair
%pip install duckdb
%pip install requests
%pip install --upgrade certifi
%pip install --upgrade requests urllib3
%pip install python-certifi-win32
%pip install python-dotenv
%pip install pyarrow
%pip install vega_datasets
%pip install itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━

In [2]:
import pandas as pd
import altair as alt
import duckdb
import requests
import certifi
import json
import os

In [3]:
alt.data_transformers.enable('vegafusion')
alt.renderers.enable('colab')

RendererRegistry.enable('colab')

In [4]:
response = requests.get("https://data360api.worldbank.org/data360/indicators?datasetId=WB_WDI")
response.status_code

200

# Identificar bases

In [5]:
url = "https://data360api.worldbank.org/data360/searchv2"
headers = {"accept": "*/*", "Content-Type": "application/json"}

all_results = []
skip = 0
top = 1000

while True:
    payload = {
        "count": True,
        "select": "series_description/database_id, series_description/database_name",
        "top": top,
        "skip": skip
    }
    r = requests.post(url, json=payload, headers=headers).json()
    results = r.get("value", [])
    if not results:
        break
    all_results.extend(results)
    skip += top
    total = r.get("@odata.count", 0)
    print(f"  Procesados: {skip}/{total}")
    if skip >= total:
        break

df_dbs = (
    pd.json_normalize(all_results)[['series_description.database_id', 'series_description.database_name']]
    .rename(columns={
        'series_description.database_id':   'database_id',
        'series_description.database_name': 'database_name'
    })
    .dropna(subset=['database_id'])
    .drop_duplicates(subset=['database_id'])
    .sort_values('database_id')
    .reset_index(drop=True)
)



  Procesados: 1000/12938
  Procesados: 2000/12938
  Procesados: 3000/12938
  Procesados: 4000/12938
  Procesados: 5000/12938
  Procesados: 6000/12938
  Procesados: 7000/12938
  Procesados: 8000/12938
  Procesados: 9000/12938
  Procesados: 10000/12938
  Procesados: 11000/12938
  Procesados: 12000/12938
  Procesados: 13000/12938


In [6]:
print(f"\nLa api da acceso a un total de {df_dbs['database_id'].nunique()} bases de datos.")
print('\nEstas son las primeras 10: \n\n', df_dbs.head(10).to_string())



La api da acceso a un total de 161 bases de datos.

Estas son las primeras 10: 

     database_id                                          database_name
0        BS_BTI  The Bertelsmann Stiftung’s Transformation Index (BTI)
1        BS_SGI                Sustainable Governance Indicators (SGI)
2  DRMKC_INFORM                                      INFORM Risk Index
3     EIA_EIAOD                 U.S. Energy Information Administration
4        EIU_DI                                        Democracy Index
5        EPC_AI                                               Epoch AI
6        FAO_AS                                               Aquastat
7      FAO_CAHD       Cost and Affordability of a Healthy Diet (CoAHD)
8        FAO_CP                                 Consumer Price Indices
9       FAO_EMS                                     Emissions Database


In [7]:

from itables import show
show(df_dbs, caption="Bases de datos Data360", pageLength=20)

Loading ITables v2.7.3 from the internet... (need help?)


In [8]:
url = "https://data360api.worldbank.org/data360/searchv2"
headers = {"accept": "*/*", "Content-Type": "application/json"}

all_results = []
skip = 0
top = 1000

while True:
    payload = {
        "count": True,
        "select": "series_description/idno, series_description/name, series_description/database_id",
        "top": top,
        "skip": skip
    }
    r = requests.post(url, json=payload, headers=headers).json()
    results = r.get("value", [])
    if not results:
        break
    all_results.extend(results)
    skip += top
    total = r.get("@odata.count", 0)
    print(f"  Procesados: {skip}/{total}")
    if skip >= total:
        break

df_idnos = (
    pd.json_normalize(all_results)[[
        'series_description.database_id',
        'series_description.idno',
        'series_description.name'
    ]]
    .rename(columns={
        'series_description.database_id': 'database_id',
        'series_description.idno':        'indicador_id',
        'series_description.name':        'indicador_name'
    })
    .dropna(subset=['indicador_id'])
    .drop_duplicates(subset=['indicador_id'])
    .sort_values('indicador_id')
    .reset_index(drop=True)
)



  Procesados: 1000/12938
  Procesados: 2000/12938
  Procesados: 3000/12938
  Procesados: 4000/12938
  Procesados: 5000/12938
  Procesados: 6000/12938
  Procesados: 7000/12938
  Procesados: 8000/12938
  Procesados: 9000/12938
  Procesados: 10000/12938
  Procesados: 11000/12938
  Procesados: 12000/12938
  Procesados: 13000/12938


In [9]:
print(f"\nLa api da acceso a un total de {df_idnos['indicador_id'].nunique()} indicadores.")
print('\nEstas son las primeras 10: \n\n', df_idnos.head(10).to_string())


La api da acceso a un total de 10327 indicadores.

Estas son las primeras 10: 

   database_id  indicador_id                                               indicador_name
0      ITU_DH    ACT_MOB_SB                  Active mobile-broadband subscriptions (ITU)
1     WB_AGCI    AG_HH_PD_V               Median Agricultural Household Production Value
2     WB_AGCI      AG_HS_CT                Median Agricultural Household Cultivated Area
3     WB_AGCI     AG_IMO_SD   Percentage of Agricultural Households using Improved Seeds
4     WB_AGCI     AG_IRG_CT       Mean Irrigated Percentage of Household Cultivated Area
5     WB_AGCI   AG_SD_HH_PD  Median Sold Percentage of Household Agricultural Production
6   WB_LPI_20         AV_DT                                   Aviation import dwell time
7   WB_LPI_20       AV_PRTN                         Number of aviation partner economies
8     WB_ID4D     BIRTH_REG             Children under age 5 whose births are registered
9      BS_BTI  BS_BTI_16_PC 

In [10]:
show(df_idnos, caption="Bases de datos Data360", pageLength=20)

Loading ITables v2.7.3 from the internet... (need help?)


## Estructura de los Datos

La API Data360 organiza la información como un **panel de datos**:
cada registro representa una observación única para una combinación de
indicador, país y año.

| Dimensión | Campo API | Descripción |
|---|---|---|
| **Base de datos** | `DATABASE_ID` | Agrupa indicadores por fuente (ej: `WB_WDI`, `WB_GS`) |
| **Indicador** | `INDICATOR` | La variable medida (ej: gasto en educación, PIB) |
| **País** | `REF_AREA` | Código ISO3 del país (ej: `ARG`, `BRA`) |
| **Período** | `TIME_PERIOD` | Año de la observación |
| **Valor** | `OBS_VALUE` | El dato numérico |

Cada base puede tener dimensiones adicionales según su temática
(por ejemplo, Gender Statistics agrega `SEX`, `AGE`, `URBANISATION`).

In [23]:
# ── Exploración general de la estructura de la API ─────────────────────────

# Tomamos WB_WDI como ejemplo representativo
url_data = "https://data360api.worldbank.org/data360/data"
r_muestra = requests.get(
    url_data,
    params={"DATABASE_ID": "WB_WDI", "INDICATOR": "WB_WDI_NY_GDP_PCAP_CD", "top": 5},
    headers={"accept": "*/*"}
).json()

df_muestra = pd.json_normalize(r_muestra.get("value", []))

print("── Ejemplo de registro crudo de la API (WB_WDI) ──")
print(df_muestra.columns.tolist())
print()
print(df_muestra.head(3).to_string(index=False))

print(f"\n── Cobertura general de la plataforma ──")
print(f"  Bases de datos disponibles : {df_dbs['database_id'].nunique()}")
print(f"  Indicadores totales        : {df_idnos['indicador_id'].nunique()}")
print(f"  Temáticas cubiertas (muestra de bases):")
print(df_dbs.head(10).to_string(index=False))

── Ejemplo de registro crudo de la API (WB_WDI) ──
['OBS_VALUE', 'TIME_FORMAT', 'UNIT_MULT', 'COMMENT_OBS', 'OBS_STATUS', 'OBS_CONF', 'AGG_METHOD', 'DECIMALS', 'COMMENT_TS', 'DATA_SOURCE', 'LATEST_DATA', 'DATABASE_ID', 'INDICATOR', 'REF_AREA', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3', 'TIME_PERIOD', 'FREQ', 'UNIT_MEASURE', 'UNIT_TYPE']

  OBS_VALUE TIME_FORMAT  UNIT_MULT COMMENT_OBS OBS_STATUS OBS_CONF AGG_METHOD DECIMALS COMMENT_TS DATA_SOURCE  LATEST_DATA DATABASE_ID             INDICATOR REF_AREA SEX AGE URBANISATION COMP_BREAKDOWN_1 COMP_BREAKDOWN_2 COMP_BREAKDOWN_3 TIME_PERIOD FREQ UNIT_MEASURE UNIT_TYPE
  697.76354         P1Y          0        None          A       PU         _Z        2       None        None        False      WB_WDI WB_WDI_NY_GDP_PCAP_CD      UZB  _T  _T           _T               _Z               _Z               _Z        1999    A          USD      None
1472.177517         P1Y          0        None          A

## Para nuestros gráficos seleccionamos estas dos bases:
### WB_WDI World Development Indicators (WDI)
### WB_GS Gender Statistics


In [11]:
url = "https://data360api.worldbank.org/data360/searchv2"
headers = {"accept": "*/*", "Content-Type": "application/json"}

all_results = []
skip = 0
top = 1000

while True:
    payload = {
        "count": True,
        "select": "series_description/database_id, series_description/idno, series_description/name",
        "filter": "series_description/database_id eq 'WB_WDI' or series_description/database_id eq 'WB_GS'",
        "top": top,
        "skip": skip
    }
    r = requests.post(url, json=payload, headers=headers).json()
    results = r.get("value", [])
    if not results:
        break
    all_results.extend(results)
    skip += top
    total = r.get("@odata.count", 0)
    print(f"  Procesados: {skip}/{total}")
    if skip >= total:
        break

df_indicators = (
    pd.json_normalize(all_results)
    .rename(columns={
        'series_description.database_id': 'database_id',
        'series_description.idno':        'indicator_id',
        'series_description.name':        'indicator_name'
    })
    [['database_id', 'indicator_id', 'indicator_name']]
    .dropna(subset=['indicator_id'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"WB_WDI: {len(df_indicators[df_indicators['database_id']=='WB_WDI'])} indicadores")
print(f"WB_GS:  {len(df_indicators[df_indicators['database_id']=='WB_GS'])} indicadores")

from itables import show
show(df_indicators, caption="Indicadores WDI y Gender Statistics", pageLength=25)

  Procesados: 1000/1898
  Procesados: 2000/1898
WB_WDI: 1533 indicadores
WB_GS:  363 indicadores


Loading ITables v2.7.3 from the internet... (need help?)


## Filtramos los indicadores que nos interesan:

In [12]:
INDICATORS = {
    "gasto_edu": "WB_WDI_SE_XPD_TOTL_GD_ZS",
    "pib_pc":    "WB_WDI_NY_GDP_PCAP_CD",
    "prim":      "WB_GS_SE_PRM_ENRR",
    "sec":       "WB_GS_SE_SEC_ENRR",
    "ter":       "WB_GS_SE_TER_ENRR",
    "comp":      "WB_GS_SE_PRM_CMPT_ZS",
}
mis_indicadores = pd.DataFrame([
    {"nombre": nombre, "indicator_id": ind_id}
    for nombre, ind_id in INDICATORS.items()
]).merge(df_indicators, on='indicator_id', how='left')

from itables import show
show(mis_indicadores, caption="Indicadores utilizados", pageLength=10)

Loading ITables v2.7.3 from the internet... (need help?)


In [13]:
def get_wb_data(indicator_id):
    db = "_".join(indicator_id.split("_")[:2])
    base_url = "https://data360api.worldbank.org/data360/data"
    headers = {"accept": "*/*"}

    all_results = []
    skip = 0
    top = 1000

    r = requests.get(
        base_url,
        params={"DATABASE_ID": db, "INDICATOR": indicator_id},
        headers=headers
    ).json()

    total = r.get("count", 0)
    all_results.extend(r.get("value", []))
    print(f"  {indicator_id}: total={total}")

    if len(all_results) >= total:
        pass
    else:
        skip += top
        while skip < total:
            r = requests.get(
                base_url,
                params={"DATABASE_ID": db, "INDICATOR": indicator_id, "skip": skip, "top": top},
                headers=headers
            ).json()
            chunk = r.get("value", [])
            if not chunk:
                break
            all_results.extend(chunk)
            skip += top
            print(f"  {indicator_id}: {min(skip, total)}/{total}")

    if not all_results:
        return pd.DataFrame()

    return pd.json_normalize(all_results).rename(columns={
        "REF_AREA":    "iso3",
        "TIME_PERIOD": "year",
        "OBS_VALUE":   "val",
    })

In [14]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

try:
    import google.colab
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    DATA_DIR = Path(
        userdata.get('DATA_DIR')
    )

except Exception:
    DATA_DIR = Path(
        os.getenv("DATA_DIR", "./data_cache")
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)

Mounted at /content/drive
data_cache


In [15]:
INDICATORS = {
    "gasto_edu": "WB_WDI_SE_XPD_TOTL_GD_ZS",
    "pib_pc":    "WB_WDI_NY_GDP_PCAP_CD",
    "prim":      "WB_GS_SE_PRM_ENRR",
    "sec":       "WB_GS_SE_SEC_ENRR",
    "ter":       "WB_GS_SE_TER_ENRR",
    "comp":      "WB_GS_SE_PRM_CMPT_ZS",
}

def load_or_fetch(name, indicator_id):
    path = os.path.join(DATA_DIR, f"{name}.parquet")
    if os.path.exists(path):
        print(f"  {name}: cargando desde disco")
        return pd.read_parquet(path)
    print(f"  {name}: descargando...")
    df = get_wb_data(indicator_id)
    df.to_parquet(path, index=False)
    return df

gasto_edu, pib_pc, prim, sec, ter, comp = [
    load_or_fetch(name, ind) for name, ind in INDICATORS.items()
]

  gasto_edu: descargando...
  WB_WDI_SE_XPD_TOTL_GD_ZS: total=6341
  WB_WDI_SE_XPD_TOTL_GD_ZS: 2000/6341
  WB_WDI_SE_XPD_TOTL_GD_ZS: 3000/6341
  WB_WDI_SE_XPD_TOTL_GD_ZS: 4000/6341
  WB_WDI_SE_XPD_TOTL_GD_ZS: 5000/6341
  WB_WDI_SE_XPD_TOTL_GD_ZS: 6000/6341
  WB_WDI_SE_XPD_TOTL_GD_ZS: 6341/6341
  pib_pc: descargando...
  WB_WDI_NY_GDP_PCAP_CD: total=14561
  WB_WDI_NY_GDP_PCAP_CD: 2000/14561
  WB_WDI_NY_GDP_PCAP_CD: 3000/14561
  WB_WDI_NY_GDP_PCAP_CD: 4000/14561
  WB_WDI_NY_GDP_PCAP_CD: 5000/14561
  WB_WDI_NY_GDP_PCAP_CD: 6000/14561
  WB_WDI_NY_GDP_PCAP_CD: 7000/14561
  WB_WDI_NY_GDP_PCAP_CD: 8000/14561
  WB_WDI_NY_GDP_PCAP_CD: 9000/14561
  WB_WDI_NY_GDP_PCAP_CD: 10000/14561
  WB_WDI_NY_GDP_PCAP_CD: 11000/14561
  WB_WDI_NY_GDP_PCAP_CD: 12000/14561
  WB_WDI_NY_GDP_PCAP_CD: 13000/14561
  WB_WDI_NY_GDP_PCAP_CD: 14000/14561
  WB_WDI_NY_GDP_PCAP_CD: 14561/14561
  prim: descargando...
  WB_GS_SE_PRM_ENRR: total=47539
  WB_GS_SE_PRM_ENRR: 2000/47539
  WB_GS_SE_PRM_ENRR: 3000/47539
  WB_GS_SE_PR

In [16]:
print(gasto_edu.head())
type(gasto_edu)
len(gasto_edu)

       val TIME_FORMAT  UNIT_MULT COMMENT_OBS OBS_STATUS OBS_CONF AGG_METHOD  \
0  6.16265         P1Y          0        None          A       PU         _Z   
1  1.77495         P1Y          0        None          A       PU         _Z   
2  3.60613         P1Y          0        None          A       PU         _Z   
3  2.46335         P1Y          0        None          A       PU         _Z   
4  2.75565         P1Y          0        None          A       PU         _Z   

  DECIMALS COMMENT_TS DATA_SOURCE  ...  SEX AGE URBANISATION COMP_BREAKDOWN_1  \
0        2       None        None  ...   _T  _T           _T               _Z   
1        2       None        None  ...   _T  _T           _T               _Z   
2        2       None        None  ...   _T  _T           _T               _Z   
3        2       None        None  ...   _T  _T           _T               _Z   
4        2       None        None  ...   _T  _T           _T               _Z   

  COMP_BREAKDOWN_2 COMP_BREAKDOW

6341

In [17]:
def get_regions():
    path = os.path.join(DATA_DIR, "reg_df.parquet")
    if os.path.exists(path):
        print("  reg_df: cargando desde Drive")
        return pd.read_parquet(path)
    print("  reg_df: descargando desde API WB...")
    reg_raw = requests.get(
        "https://api.worldbank.org/v2/country?format=json&per_page=1000"
    ).json()
    reg_df = pd.json_normalize(reg_raw[1])[['id', 'region.value']].rename(
        columns={'id': 'iso3', 'region.value': 'region'}
    )
    reg_df = reg_df[reg_df['region'] != 'Aggregates']
    reg_df.to_parquet(path, index=False)
    print("  reg_df: guardado")
    return reg_df

reg_df = get_regions()

  reg_df: descargando desde API WB...
  reg_df: guardado


In [18]:
df_viz1 = gasto_edu.merge(pib_pc, on=['iso3', 'year']).merge(reg_df, on='iso3')
df_viz1['year'] = pd.to_numeric(df_viz1['year'])
df_viz1 = df_viz1.dropna(subset=['val_x', 'val_y'])
df_viz1 = df_viz1.rename(columns={'country.value_x': 'country_name'})
slider = alt.binding_range(min=2000, max=2022, step=1, name='Año: ')
select_year = alt.selection_point(fields=['year'], bind=slider, value=2019, name="Year")

chart = alt.Chart(df_viz1).mark_circle(size=100, opacity=0.7).encode(
    x=alt.X("val_y:Q", title="PIB per cápita (USD)", scale=alt.Scale(type='log'), axis=alt.Axis(grid=False)),
    y=alt.Y("val_x:Q", title="Gasto en Educación (% PIB)", scale=alt.Scale(domain=[0, 15])),
    color=alt.Color("region:N", title="Región"),
    tooltip=[
        alt.Tooltip("iso3:N", title="País"),
        alt.Tooltip("year:O", title="Año"),
        alt.Tooltip("val_y:Q", title="PIB p/c", format="$,.0f"),
        alt.Tooltip("val_x:Q", title="% Gasto", format=".1f")
    ]
).add_params(select_year).transform_filter(select_year).properties(
    width=600, height=400, title="Inversión Educativa vs Riqueza (Escala Logarítmica)"
).interactive()

chart

alt.Chart(...)

In [19]:
import pandas as pd
import altair as alt

# 1. Concatenación
df_traj = pd.concat([
    prim[['iso3', 'year', 'val', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1']].assign(nivel="Primaria"),
    sec[['iso3', 'year', 'val', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1']].assign(nivel="Secundaria"),
    ter[['iso3', 'year', 'val', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1']].assign(nivel="Terciaria")
])

# 2. Filtrado CORREGIDO (Aquí está la magia)
df_traj = df_traj[
    (df_traj['SEX'] == '_T') &
    (df_traj['AGE'] == '_T') &
    (df_traj['URBANISATION'] == '_T') &
    (df_traj['COMP_BREAKDOWN_1'].isin(['GS_DXT', '_Z'])) # <--- Ahora acepta Terciaria
]

# 3. Creación del subset para Argentina
df_arg_traj = (
    df_traj[df_traj['iso3'] == 'ARG']
    .dropna(subset=['val'])
    [['iso3', 'year', 'val', 'nivel']]
    .copy()
)

# Convertir a numérico
df_arg_traj['year'] = pd.to_numeric(df_arg_traj['year'])
df_arg_traj['val']  = pd.to_numeric(df_arg_traj['val'])

In [20]:
# =========================
# Gobiernos y partidos
# =========================

gobiernos = pd.DataFrame({
    'year_start': [
        1970, 1971, 1973, 1973, 1974,
        1976, 1983, 1989, 1999, 2001,
        2002, 2003, 2007, 2015, 2019
    ],
    'year_end': [
        1971, 1973, 1973, 1974, 1976,
        1983, 1989, 1999, 2001, 2002,
        2003, 2007, 2015, 2019, 2022
    ],
    'presidente': [
        'Levingston', 'Lanusse', 'Cámpora', 'Perón', 'Isabel Perón',
        'Dictadura Militar', 'Alfonsín', 'Menem', 'De la Rúa',
        'Rodríguez Saá', 'Duhalde', 'Kirchner', 'CFK',
        'Macri', 'Alberto Fernández'
    ],
    'partido': [
        'De Facto', 'De Facto', 'Peronismo', 'Peronismo', 'Peronismo',
        'De Facto', 'UCR', 'Peronismo', 'UCR',
        'Peronismo', 'Peronismo', 'Peronismo', 'Peronismo',
        'PRO', 'Peronismo'
    ]
})

# =========================
# Hitos educativos
# =========================

hitos_educativos = pd.DataFrame({
    'year': [1993, 2006],
    'hito': [
        'Ley Federal\nde Educación',
        'Ley de Educación\nNacional'
    ]
})

# =========================
# Selecciones interactivas
# =========================

seleccion_nivel = alt.selection_point(
    fields=['nivel'],
    bind='legend'
)

seleccion_partido = alt.selection_point(
    fields=['partido'],
    bind='legend'
)

# =========================
# Fondos por gobierno
# =========================

rangos = (
    alt.Chart(gobiernos)
    .mark_rect(
        clip=True
    )
    .encode(
        x=alt.X("year_start:Q"),
        x2="year_end:Q",
        color=alt.Color(
            "partido:N",
            scale=alt.Scale(
                domain=['Peronismo', 'UCR', 'PRO', 'De Facto'],
                range=['#2E74C0', '#C62828', '#F9A825', '#616161']
            ),
            legend=alt.Legend(title="Espacio político")
        ),
        opacity=alt.condition(
            seleccion_partido,
            alt.value(0.18),
            alt.value(0.04)
        ),
        tooltip=[
            alt.Tooltip("presidente:N", title="Gobierno"),
            alt.Tooltip("partido:N", title="Partido")
        ]
    )
    .add_params(seleccion_partido)
)

# =========================
# Reglas verticales
# =========================

reglas = (
    alt.Chart(gobiernos)
    .mark_rule(
        strokeDash=[6, 4],
        strokeWidth=1,
        color='gray',
        opacity=0.25,
        clip=True
    )
    .encode(
        x=alt.X("year_start:Q")
    )
)

# =========================
# Hitos educativos
# =========================

lineas_hitos = (
    alt.Chart(hitos_educativos)
    .mark_rule(
        color='black',
        strokeWidth=2,
        strokeDash=[4, 4],
        opacity=0.8,
        clip=True
    )
    .encode(
        x=alt.X("year:Q"),
        tooltip=[
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("hito:N", title="Hito")
        ]
    )
)

texto_hitos = (
    alt.Chart(hitos_educativos)
    .mark_text(
        angle=0,
        align='left',
        baseline='bottom',
        lineBreak='\n',
        dx=5,
        dy=-5,
        color='blue',
        fontWeight='bold',
        fontSize=11,
        clip=True
    )
    .encode(
        x=alt.X("year:Q"),
        y=alt.value(390),
        text=alt.Text("hito:N")
    )
)

# =========================
# Trayectorias
# =========================

chart_traj = (
    alt.Chart(df_arg_traj)
    .mark_line(point=True)
    .encode(
        x=alt.X(
            "year:Q",
            title="Año",
            axis=alt.Axis(format='d', tickCount=10, grid=False),
            scale=alt.Scale(zero=False, domain=[1970, 2022])
        ),
        y=alt.Y(
            "val:Q",
            title="Tasa de matrícula (% bruta)",
            scale=alt.Scale(zero=False)
        ),
        color=alt.Color(
            "nivel:O",
            title="Nivel Educativo",
            sort=['Primaria', 'Secundaria', 'Terciaria']
        ),
        opacity=alt.condition(
            seleccion_nivel,
            alt.value(1),
            alt.value(0.15)
        ),
        size=alt.condition(
            seleccion_nivel,
            alt.value(3),
            alt.value(1.5)
        ),
        tooltip=[
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("nivel:N", title="Nivel"),
            alt.Tooltip("val:Q", title="Tasa (%)", format=".2f")
        ]
    )
    .add_params(seleccion_nivel)
)

# =========================
# Gráfico final
# =========================

final_chart = (
    rangos
    + reglas
    + lineas_hitos
    + texto_hitos
    + chart_traj
).resolve_scale(
    x='shared',
    color='independent'
).properties(
    width=700,
    height=400,
    title="Evolución de la Matrícula por Nivel Educativo en Argentina"
)

final_chart.display()

alt.LayerChart(...)

In [21]:
from vega_datasets import data
iso_map = pd.read_csv("https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv")[["alpha-3", "country-code"]].rename(columns={"alpha-3": "iso3", "country-code": "id_num"})
df_map = comp[comp['year'] == '2020'].merge(iso_map, on='iso3')

world = alt.topo_feature(data.world_110m.url, 'countries')

mapa_base = alt.Chart(world).mark_geoshape(
    fill='lightgray',
    stroke='white',
    strokeWidth=0.5
).project('naturalEarth1')

mapa_datos = alt.Chart(world).mark_geoshape().encode(
    color=alt.Color(
        'val:Q',
        title='% Finalización',
        scale=alt.Scale(scheme='viridis', domain=[50, 100])
    ),
    tooltip=[
        alt.Tooltip('iso3:N', title='Código'),
        alt.Tooltip('val:Q', title='% Tasa', format='.2f')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(df_map, 'id_num', ['val', 'iso3'])
).project('naturalEarth1')

mapa_final = (mapa_base + mapa_datos).properties(
    width=700,
    height=400,
    title="Acceso Universal: Tasa de Finalización Primaria (2020)"
)

mapa_final.display()

alt.LayerChart(...)

## Estructura de Datos

La API Data360 organiza la información según las siguientes dimensiones principales:

| Dimensión | Descripción |
|---|---|
| **Base de datos** (`database_id`) | Agrupa series por fuente/programa (ej: `WB_WDI`, `WB_GS`) |
| **Indicador** (`indicator_id` / `idno`) | Identifica la variable medida |
| **País** (`REF_AREA`) | Código ISO3 del país o agregado regional |
| **Período** (`TIME_PERIOD`) | Año de observación (cobertura variable por indicador) |
| **Valor** (`OBS_VALUE`) | El dato numérico de la observación |

Cada registro es, en esencia, una tupla **(indicador, país, año) → valor**,
lo que define una estructura de **panel de datos** (o *long format*).

In [22]:
# ── Estructura de datos: análisis exploratorio ─────────────────────────────

print("=" * 55)
print("DIMENSIONES DE LOS DATOS")
print("=" * 55)

# Cobertura temporal por indicador
print("\n Cobertura temporal por indicador:")
for nombre, df in [("gasto_edu", gasto_edu), ("pib_pc", pib_pc),
                   ("prim", prim), ("sec", sec), ("ter", ter), ("comp", comp)]:
    años = df["year"].dropna().astype(int)
    print(f"  {nombre:12s}: {años.min()} – {años.max()}  ({años.nunique()} años, {len(df):,} obs)")

# Cobertura geográfica
print("\n Cobertura geográfica (países únicos por indicador):")
for nombre, df in [("gasto_edu", gasto_edu), ("pib_pc", pib_pc),
                   ("prim", prim), ("sec", sec), ("ter", ter), ("comp", comp)]:
    print(f"  {nombre:12s}: {df['iso3'].nunique():3d} países")

# Distribución regional (usando reg_df ya construido)
print("\n Distribución por región (sobre df_viz1):")
print(df_viz1.groupby("region")["iso3"].nunique().sort_values(ascending=False).to_string())

# Completitud / valores disponibles
print("\n Completitud de datos (% de obs con valor no nulo):")
for nombre, df in [("gasto_edu", gasto_edu), ("pib_pc", pib_pc),
                   ("prim", prim), ("sec", sec), ("ter", ter), ("comp", comp)]:
    pct = df["val"].notna().mean() * 100
    print(f"  {nombre:12s}: {pct:.1f}%")

# Esquema de columnas de un DataFrame representativo
print("\n Esquema de columnas (gasto_edu como ejemplo):")
print(gasto_edu.dtypes.to_string())
print(f"\nPrimeras filas:")
print(gasto_edu.head(3).to_string(index=False))

DIMENSIONES DE LOS DATOS

 Cobertura temporal por indicador:
  gasto_edu   : 1970 – 2025  (56 años, 6,341 obs)
  pib_pc      : 1960 – 2024  (65 años, 14,561 obs)
  prim        : 1970 – 2022  (53 años, 47,539 obs)
  sec         : 1970 – 2022  (53 años, 38,965 obs)
  ter         : 1970 – 2022  (53 años, 24,079 obs)
  comp        : 1970 – 2022  (53 años, 21,560 obs)

 Cobertura geográfica (países únicos por indicador):
  gasto_edu   : 248 países
  pib_pc      : 262 países
  prim        : 257 países
  sec         : 257 países
  ter         : 250 países
  comp        : 251 países

 Distribución por región (sobre df_viz1):
region
Europe & Central Asia                                51
Sub-Saharan Africa                                   48
Latin America & Caribbean                            39
East Asia & Pacific                                  32
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                   